# ``FCNet`` regression demo.

In this demo, we'll use the ``FCNet`` model in a regression problem pertaining to red wine!


## The data

The data are availe [here](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009), on Kaggle. Download them, and in particular, grab the csv file ``winequality-red.csv``.


First, we load and take a look at them.

In [1]:
from pathlib import Path
from pandas import read_csv


data_path = Path("/home/jim/Downloads/winequality-red.csv")

data = read_csv(data_path)
print(data.keys())

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')


All of the features are continuous variables except the ``"quality"`` field, which we'll exclude.

In [2]:
data = data.drop(labels=["quality"], axis=1)

(1599, 11)


Now we split the data for training and validation.

In [6]:
valid_split = data.sample(frac=0.25, random_state=123)
train_split = data.loc[~data.index.isin(valid_split.index)]

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
3              11.2             0.280         0.56             1.9      0.075   
4               7.4             0.700         0.00             1.9      0.076   
...             ...               ...          ...             ...        ...   
1592            6.3             0.510         0.13             2.3      0.076   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1596            6.3             0.510         0.13             2.3      0.076   
1597            5.9             0.645         0.12             2.0      0.075   

      free sulfur dioxide  

Next, we rescale the data using the _training_ mean and standard deviation

In [10]:
means = train_split.mean()
devs = train_split.std()


train_split = (train_split - means) / devs
valid_split = (valid_split - means) / devs

fixed acidity           0.000000e+00
volatile acidity        0.000000e+00
citric acid             0.000000e+00
residual sugar          1.185226e-17
chlorides               0.000000e+00
free sulfur dioxide     1.185226e-17
total sulfur dioxide   -2.370451e-17
density                 0.000000e+00
pH                      0.000000e+00
sulphates               0.000000e+00
alcohol                -2.370451e-17
dtype: float64
fixed acidity           1.0
volatile acidity        1.0
citric acid             1.0
residual sugar          1.0
chlorides               1.0
free sulfur dioxide     1.0
total sulfur dioxide    1.0
density                 1.0
pH                      1.0
sulphates               1.0
alcohol                 1.0
dtype: float64


Now the training means should be precisely zero and the standard devaitions one.

In [11]:
print(train_split.mean())
print(train_split.std())

fixed acidity           0.000000e+00
volatile acidity        0.000000e+00
citric acid             0.000000e+00
residual sugar          1.185226e-17
chlorides               0.000000e+00
free sulfur dioxide     1.185226e-17
total sulfur dioxide   -2.370451e-17
density                 0.000000e+00
pH                      0.000000e+00
sulphates               0.000000e+00
alcohol                -2.370451e-17
dtype: float64
fixed acidity           1.0
volatile acidity        1.0
citric acid             1.0
residual sugar          1.0
chlorides               1.0
free sulfur dioxide     1.0
total sulfur dioxide    1.0
density                 1.0
pH                      1.0
sulphates               1.0
alcohol                 1.0
dtype: float64


Now we decide what we're going to predict, and what we're going to use as input features.

In [15]:
target_key = "density"
feature_keys = list(filter(lambda x: x != target_key, train_split.keys()))

Now we create datasets

In [29]:
from torch_tools import DataSet

from torch import from_numpy

from torchvision.transforms import Compose


def transforms(training: bool = False) -> Compose:
    """Return a composition of transforms for the inputs.

    Parameters
    ----------
    training : bool, optional
        Are these training transforms? If ``False``, we assume validation.

    Returns
    -------

    """
    tfm_list = [lambda x: from_numpy(x).float()]

    return Compose(tfm_list)


train_set = DataSet(
    inputs=tuple(train_split[feature_keys].to_numpy()),
    input_tfms=transforms(),
    targets=tuple(train_split[[target_key]].to_numpy()),
    target_tfms=transforms(),
)

valid_set = DataSet(
    inputs=tuple(valid_split[feature_keys].to_numpy()),
    input_tfms=transforms(),
    targets=tuple(valid_split[[target_key]].to_numpy()),
    target_tfms=transforms(),
)
